# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aneeqahabib/FlyRank_ML_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

The main traffic and freshness fields are strongly heavy-tailed. A few pages have very large impression totals while most are small, so I compare medians and tail ratios instead of trusting raw means. The clearest pattern in this lane is that ranking position separates CTR far more than search volume or age do.


In [1]:
import pandas as pd
import numpy as np

# Load the same starter slice used throughout the lane.
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = df['trend_direction'].eq('down').astype(int)

print('Rows:', len(df))
print('Declining rate:', round(df['is_declining_label'].mean(), 3))
print('\nTraffic distributions (heavy tails):')
for col in ['impressions_90d', 'clicks_90d', 'search_volume', 'days_since_last_update']:
    s = df[col].dropna()
    print(f"{col:>22}: median={s.median():,.1f}, p95={s.quantile(0.95):,.1f}, max={s.max():,.1f}")

# Visibility-first check: ranking position and CTR on a meaningful-volume slice.
visible = df[df['impressions_90d'] >= 100].copy()
position_ctr = (
    visible.groupby('position_tier', observed=False)['ctr']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
    .round(4)
)
print('\nCTR by position tier for pages with >=100 impressions:')
print(position_ctr)

# Staleness and demand are noisier once we respect the heavy-tail structure.
print('\nSearch-volume vs impressions (Spearman):')
print(round(df[['search_volume', 'impressions_90d']].dropna().corr(method='spearman').iloc[0, 1], 4))


Rows: 30000
Declining rate: 0.542

Traffic distributions (heavy tails):
       impressions_90d: median=731.0, p95=22,996.5, max=517,715.0
            clicks_90d: median=1.0, p95=69.0, max=4,178.0
         search_volume: median=10.0, p95=390.0, max=74,000.0
days_since_last_update: median=20.0, p95=104.0, max=373.0

CTR by position tier for pages with >=100 impressions:
                 mean  count
position_tier               
page_1         0.3548   8633
top_3          0.3341    533
striking       0.2558   5903
page_3_5       0.1424   6058
deep           0.0554    879

Search-volume vs impressions (Spearman):
-0.0291


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
# Signal test 1: ranking position is associated with CTR.
visible = df[df['impressions_90d'] >= 100].copy()
position_table = (
    visible.groupby('position_tier', observed=False)
    .agg(n=('content_id', 'size'), mean_ctr=('ctr', 'mean'))
    .reindex(['top_3', 'page_1', 'striking', 'page_3_5', 'deep'])
    .reset_index()
)
print('Signal test 1 — position tier and CTR')
print(position_table.to_string(index=False, formatters={'mean_ctr': lambda x: f'{x:.4f}'}))

# Signal test 2: search volume is not a strong standalone signal.
search_corr = df[['search_volume', 'impressions_90d']].dropna().corr(method='spearman').iloc[0, 1]
print('\nSignal test 2 — search_volume vs impressions_90d (Spearman):', round(search_corr, 4))

# Signal test 3: staleness is a directionally useful but weak signal.
stale_bucket = pd.cut(
    df['days_since_last_update'],
    bins=[-np.inf, 90, 180, 365, np.inf],
    labels=['0-90', '91-180', '181-365', '365+'],
)
staleness_table = (
    df.assign(stale_bucket=stale_bucket)
    .groupby('stale_bucket', observed=False)
    .agg(n=('content_id', 'size'), decline_rate=('is_declining_label', 'mean'))
    .reset_index()
)
print('\nSignal test 3 — staleness bucket and observed decline rate')
print(staleness_table.round(4).to_string(index=False))

# Mini-verdicts
print('\nVerdicts:')
# 1) ranking position: economic signal, stronger than volume.
print('1) Position tier on visible pages: CONFIRMED. Pages that are farther down the rankings have much lower CTR, even when restricting to non-noise pages with >=100 impressions.')
# 2) search volume: weak and near zero. Use as context, not as a top-level action rule.
print('2) Search volume vs impressions: MIXED. The Spearman correlation is close to zero, so keyword demand is not a strong standalone predictor in this slice.')
# 3) staleness: relevant but not overwhelming; older pages are not universally declining in this slice.
print('3) Staleness bucket: MIXED. The oldest bucket is not uniformly worse, and the count in 181+ day buckets is tiny, so age is a useful review cue rather than a decisive signal.')


Signal test 1 — position tier and CTR
position_tier    n mean_ctr
        top_3  533   0.3341
       page_1 8633   0.3548
     striking 5903   0.2558
     page_3_5 6058   0.1424
         deep  879   0.0554

Signal test 2 — search_volume vs impressions_90d (Spearman): -0.0291

Signal test 3 — staleness bucket and observed decline rate
stale_bucket     n  decline_rate
        0-90 20655        0.5120
      91-180  9171        0.6111
     181-365   169        0.4675
        365+     5        0.6000

Verdicts:
1) Position tier on visible pages: CONFIRMED. Pages that are farther down the rankings have much lower CTR, even when restricting to non-noise pages with >=100 impressions.
2) Search volume vs impressions: MIXED. The Spearman correlation is close to zero, so keyword demand is not a strong standalone predictor in this slice.
3) Staleness bucket: MIXED. The oldest bucket is not uniformly worse, and the count in 181+ day buckets is tiny, so age is a useful review cue rather than a decis

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# Flag-linked test: does the stale + visible rule assumption hold?
# This rule is a real FlyRank-style review heuristic: stale pages with prior visibility get extra attention.
flag_frame = (
    df.assign(
        stale_180=df['days_since_last_update'] >= 180,
        visible_100=df['impressions_prev_30d'] >= 100,
    )
    .groupby(['stale_180', 'visible_100'], observed=False)
    .agg(n=('content_id', 'size'), observed_decline_rate=('is_declining_label', 'mean'))
    .reset_index()
)
print('Flag-linked test: stale pages with prior visibility')
print(flag_frame.round(4).to_string(index=False))

# We only read the bucket that matches the rule's assumption.
rule_bucket = flag_frame[(flag_frame['stale_180'] == True) & (flag_frame['visible_100'] == True)]
other_bucket = flag_frame[(flag_frame['stale_180'] == False) & (flag_frame['visible_100'] == True)]

print('\nRule-support check: stale + visible vs. non-stale + visible')
print('Rule bucket observed decline rate:', round(rule_bucket['observed_decline_rate'].iat[0], 4), 'n=', int(rule_bucket['n'].iat[0]))
print('Control bucket observed decline rate:', round(other_bucket['observed_decline_rate'].iat[0], 4), 'n=', int(other_bucket['n'].iat[0]))

# Sample-size floor: tiny buckets are insufficient.
if int(rule_bucket['n'].iat[0]) < 50:
    print('Verdict: FALSE / insufficient data. The stale + visible bucket is too small to support a strong rule claim in this slice.')
else:
    if float(rule_bucket['observed_decline_rate'].iat[0]) > float(other_bucket['observed_decline_rate'].iat[0]):
        print('Verdict: CONFIRMED. The stale + visible slice has a higher observed decline rate than the non-stale + visible control, which supports the rule assumption.')
    else:
        print('Verdict: MIXED. The stale + visible slice does not clearly outperform the control, so the rule is only directionally useful.')


Flag-linked test: stale pages with prior visibility
 stale_180  visible_100     n  observed_decline_rate
     False        False 11837                 0.4317
     False         True 17989                 0.6154
      True        False   153                 0.4052
      True         True    21                 0.9524

Rule-support check: stale + visible vs. non-stale + visible
Rule bucket observed decline rate: 0.9524 n= 21
Control bucket observed decline rate: 0.6154 n= 17989
Verdict: FALSE / insufficient data. The stale + visible bucket is too small to support a strong rule claim in this slice.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print('What this means in practice:')
print('A content team should treat ranking position as the clearest operational signal in this lane, even after filtering out noise. Search volume by itself is too weak to justify a demand-first strategy, and staleness is useful for triage but should not be treated as a standalone health judgment.')
print('For decision support, the best use of these signals is to prioritize pages for review: look first at pages with weak visibility and then add freshness or recency checks only as a second screen, not as a causal explanation.')


What this means in practice:
A content team should treat ranking position as the clearest operational signal in this lane, even after filtering out noise. Search volume by itself is too weak to justify a demand-first strategy, and staleness is useful for triage but should not be treated as a standalone health judgment.
For decision support, the best use of these signals is to prioritize pages for review: look first at pages with weak visibility and then add freshness or recency checks only as a second screen, not as a causal explanation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.